# Parsing Milestone 2 theo thành viên trên Colab CPU

Mỗi thành viên chạy notebook này trên dữ liệu MS1 thuộc dải ID mình đã được giao. Chọn **Runtime → Change runtime type → CPU**. Cả nhóm dùng cùng commit code được cố định trong notebook.

Đầu vào phải có discovery, checkpoint download/extract, raw, corpus và các sidecar; dừng tiến trình MS1 đang ghi trước khi chạy. ZIP log/benchmark MS1 không thay thế dữ liệu nguồn.

Mỗi người chạy thử `SMOKE` trước, sau đó chuyển sang `FULL`. Các thành viên có thể chạy song song trên output riêng. Không mở hai runtime ghi cùng output của một thành viên.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Chọn thành viên và dữ liệu

Chọn `MEMBER` từ TV1 đến TV5, rồi nhập `MS1_INPUT` là thư mục data MS1 của mình. Để trống thì dùng `/content/drive/MyDrive/arxiv_ms1_data/<member>/data`. Nếu dùng thư mục được chia sẻ, thêm shortcut vào My Drive và nhập đúng đường dẫn nhìn thấy trên Colab.

`SMOKE` chạy tối đa ba bài đã discovery thành công trong dải của mình; có thể nhập ID mẫu ở `SMOKE_IDS`. `FULL` xử lý đủ dải 5.000 ID, ghi nhận cả ID chưa discovery hoặc thiếu source/checkpoint. Không cần tự sửa range hay danh sách `JOBS`.

Giữ `RUN_TAG`, thành viên và output root để resume. Muốn chạy một đợt mới độc lập thì đổi tag. SMOKE và FULL luôn dùng hai output khác nhau. Chỉ bật `VERIFY_RESUME` trong lượt FULL nếu muốn đọc lại toàn bộ đầu vào để kiểm tra cache; SMOKE luôn kiểm tra bước này.

In [ ]:
from pathlib import Path
# Mỗi thành viên chỉ sửa các lựa chọn trong cell này.
MEMBER = 'tv1' #@param ["tv1", "tv2", "tv3", "tv4", "tv5"]
RUN_MODE = 'SMOKE' #@param ["SMOKE", "FULL"]
MS1_INPUT = '' #@param {type:"string"}
OUTPUT_ROOT = '/content/drive/MyDrive/arxiv_ms2_results' #@param {type:"string"}
RUN_TAG = 'ms2-members-20261110' #@param {type:"string"}
SMOKE_IDS = '' #@param {type:"string"}
VERIFY_RESUME = False #@param {type:"boolean"}
# Cùng mã production đã nghiệm thu; không đổi riêng từng người.
CODE_REF = 'b932d98eae90d50611004ccc5e2d91709a77b7bf'
RESUME = True

In [ ]:
import subprocess, sys
REPO = Path('/content/arxiv-ms2-integration')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/LeoTkiet/arxiv-data-science.git', str(REPO)], check=True)
if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
    raise RuntimeError('Checkout has local changes; keep them before changing CODE_REF.')
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', CODE_REF], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
CODE_SHA = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO) + '[reporting]'], check=True)
# Add the checked-out source even when this runtime previously installed another checkout.
sys.path.insert(0, str(REPO / 'src'))
print('Commit:', CODE_SHA)

## 2. Xác nhận phạm vi của mình

Notebook đọc phân công từ `configs/ms1_group6_ranges.json` trong cùng checkout. Kiểm tra thành viên, input, output và range in dưới đây trước khi chạy cell parsing.

| Thành viên | Dải ID |
| --- | --- |
| TV1 | 2508.12872–2508.17871 |
| TV2 | 2506.15788–2506.20787 |
| TV3 | 2411.08799–2411.13798 |
| TV4 | 2410.18025–2410.23024 |
| TV5 | 2410.03025–2410.08024 |

In [ ]:
import json, re
from arxiv_pipeline.main import expand_range
from arxiv_pipeline.pipeline import parse

MEMBER_ROWS = {'tv1': 15, 'tv2': 24, 'tv3': 56, 'tv4': 59, 'tv5': 62}
if MEMBER not in MEMBER_ROWS or RUN_MODE not in {'SMOKE', 'FULL'}:
    raise ValueError('Chọn đúng MEMBER và RUN_MODE trong danh sách.')
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]*', RUN_TAG):
    raise ValueError('RUN_TAG chỉ dùng chữ, số, dấu chấm, gạch dưới hoặc gạch nối.')
assignment_config = json.loads((REPO / 'configs/ms1_group6_ranges.json').read_text())
matches = [r for r in assignment_config['ranges'] if r['source_row'] == MEMBER_ROWS[MEMBER]]
if len(matches) != 1:
    raise ValueError('Không xác định được dải phân công từ configs/ms1_group6_ranges.json.')
assignment = matches[0]
assigned_ids = expand_range(assignment['range'])
if len(assigned_ids) != assignment['count']:
    raise ValueError('Dải phân công không khớp số ID được giao.')
input_root = Path(MS1_INPUT.strip() or f'/content/drive/MyDrive/arxiv_ms1_data/{MEMBER}/data')
OUTPUT_BASE = Path(OUTPUT_ROOT) / RUN_TAG / RUN_MODE.lower()
parse.validate_roots(input_root, OUTPUT_BASE / MEMBER)
if RUN_MODE == 'FULL':
    selected = None
elif SMOKE_IDS.strip():
    selected = SMOKE_IDS.replace(',', ' ').split()
else:
    records, errors, _ = parse._discover(input_root.resolve(), None)
    if errors:
        raise ValueError(f'Discovery chưa đọc được: {errors[:3]}')
    selected = [record['arxiv_id'] for record, _, conflict in records
                if not conflict and record.get('status') == 'success'
                and record['arxiv_id'] in set(assigned_ids)][:3]
if selected is not None and (not selected or not set(selected).issubset(assigned_ids)):
    raise ValueError('Cần ID mẫu hợp lệ thuộc dải của thành viên; kiểm tra discovery hoặc SMOKE_IDS.')
JOBS = [dict(member=MEMBER, input=str(input_root), ids=selected,
             total_assigned=assignment['count'], assignment_range=assignment['range'])]
CHECK_RESUME = RUN_MODE == 'SMOKE' or VERIFY_RESUME
print('Thành viên:', MEMBER, '| chế độ:', RUN_MODE, '| range:', assignment['range'])
print('Số ID sẽ xử lý:', len(assigned_ids) if selected is None else len(selected))
print('Input:', input_root)
print('Output:', OUTPUT_BASE / MEMBER)
print('Commit:', CODE_SHA)

## 3. Chạy parsing và kiểm tra đầu ra

Cell này có thể chạy lâu trong chế độ FULL. Nếu Colab bị ngắt, kết nối lại và chạy cùng cấu hình với RESUME=True để tiếp tục checkpoint. Không bật MS1 crawl trên cùng input trong lúc parsing.


In [ ]:
import json
from collections import Counter
from arxiv_pipeline.common.utils import atomic_json
from arxiv_pipeline.main import expand_range
from arxiv_pipeline.pipeline import parse
from arxiv_pipeline.export.benchmark import execute_pipeline_benchmark

members = [job['member'] for job in JOBS]
if not JOBS or len(set(members)) != len(members):
    raise ValueError('JOBS must contain distinct member names.')
for job in JOBS:
    if job['member'] not in {'tv1', 'tv2', 'tv3', 'tv4', 'tv5'}:
        raise ValueError('Unknown member name.')
    if type(job['total_assigned']) is not int or job['total_assigned'] < 1:
        raise ValueError('total_assigned must be a positive integer.')
    if job['ids'] is not None and (not isinstance(job['ids'], list) or not job['ids']):
        raise ValueError('Use a nonempty ID list for a sample, or None for all records.')
    assigned = set(expand_range(job['assignment_range']))
    if len(assigned) != job['total_assigned']:
        raise ValueError('Assignment range and denominator differ.')
    if job['ids'] is not None and not set(job['ids']).issubset(assigned):
        raise ValueError('Selected sample is outside the assigned range.')
    parse.validate_roots(job['input'], OUTPUT_BASE / job['member'])
for job in JOBS:
    for other in JOBS:
        parse.validate_roots(job['input'], OUTPUT_BASE / other['member'])

results = []
for job in JOBS:
    output = OUTPUT_BASE / job['member']
    output.mkdir(parents=True, exist_ok=True)
    # FULL selects every assigned ID explicitly, including missing discovery.
    identifiers = expand_range(job['assignment_range']) if job['ids'] is None else job['ids']
    print('Starting', job['member'], 'IDs:', len(identifiers), '| output:', output, flush=True)
    atomic_json(output / 'acceptance-config.json', dict(job=job, code_sha=CODE_SHA, resume=RESUME))
    benchmark_error = None
    try:
        execute_pipeline_benchmark(
            job['input'], output, identifiers=identifiers, resume=RESUME,
            total_assigned=job['total_assigned'],
            summary_out=output / 'summaries/parsing_summary.json')
    except Exception as exc:
        # Leave the stage evidence available for diagnosis; do not reuse old metrics.
        benchmark_error = f'{type(exc).__name__}: {exc}'
        atomic_json(output / 'summaries/benchmark-error.json', dict(error=benchmark_error))
    if benchmark_error:
        result = dict(member=job['member'], code_sha=CODE_SHA, complete=False,
                      full_assignment_accepted=False, benchmark_error=benchmark_error)
        atomic_json(output / 'acceptance.json', result)
        results.append(result)
        print(job['member'], result)
        continue

    summary = json.loads((output / 'parsing_summary.json').read_text())
    stages = json.loads((output / 'manifests/stages.json').read_text())
    atomic_json(output / 'summaries/parse-run.json', summary)
    issues = []
    for stage in stages:
        if stage['status'] not in parse.TERMINAL:
            issues.append(dict(stage=stage['stage'], paper_id=stage['paper_id'],
                               version=stage['version'], status=stage['status'], error=stage.get('error')))
        if stage['stage'] == 'validate_parse' and stage.get('artifacts', {}).get('validation'):
            report = json.loads((output / stage['artifacts']['validation']).read_text())
            issues.extend(dict(stage='validate_parse', **issue) for issue in report['errors'] + report['warnings'])
    resume_events = []
    resume_ok = None
    if CHECK_RESUME:
        resumed = parse.run(identifiers, output, input_root=Path(job['input']),
                            resume=True, observer=resume_events.append)
        atomic_json(output / 'summaries/resume-events.json', resume_events)
        resume_ok = bool(summary['complete'] and resumed['complete'] and resume_events
                         and all(event['cached'] for event in resume_events))

    outcomes = summary['paper_results']
    excluded = summary['excluded_inputs']
    accounted = {p['arxiv_id'] for p in outcomes
                 if p.get('reason') != 'MS1 discovery record is missing'} | {
        p.get('arxiv_id') or p.get('input_id') for p in excluded}
    assigned = set(expand_range(job['assignment_range']))
    result = dict(
        member=job['member'], code_sha=CODE_SHA, resume_enabled=RESUME,
        scope='all_assigned_ids' if job['ids'] is None else 'selected_sample',
        selected_ids=job['ids'], total_assigned=job['total_assigned'],
        attempted_papers=len(outcomes), accounted_inputs=len(accounted),
        unseen_assigned_ids=sorted(assigned - accounted),
        out_of_range_ids=sorted(accounted - assigned),
        complete=summary['complete'], input_errors=summary['input_errors'],
        paper_statuses=dict(Counter(p['status'] for p in outcomes)),
        version_statuses=dict(Counter(v['status'] for p in outcomes for v in p['versions'])),
        incomplete_papers=[p['arxiv_id'] for p in outcomes if not p['complete']],
        resume_all_cached=resume_ok, issues=issues,
        full_assignment_accepted=bool(
            job['ids'] is None and summary['complete'] and not summary['input_errors']
            and accounted == assigned
            and (not CHECK_RESUME or resume_ok)))
    atomic_json(output / 'acceptance.json', result)
    results.append(result)
    print(job['member'], {k: v for k, v in result.items() if k not in {'issues', 'selected_ids', 'unseen_assigned_ids', 'out_of_range_ids', 'incomplete_papers'}})

print('Evidence:', OUTPUT_BASE / JOBS[0]['member'])

## 4. Bàn giao kết quả

Kết quả nằm trong `<OUTPUT_ROOT>/<RUN_TAG>/<smoke|full>/<MEMBER>/`. Sau khi thử SMOKE, đổi `RUN_MODE='FULL'`, chạy lại cell cấu hình phạm vi và cell parsing. Không cần chạy lại mount hoặc checkout khi chỉ đổi chế độ. FULL có đủ dải ID; bài chưa sẵn sàng sẽ được báo lỗi/incomplete, không bỏ qua để đạt complete.

Gửi output FULL hoặc ZIP ở cell cuối cho TV1 tích hợp và TV5 thống kê/báo cáo. Giữ đủ `acceptance-config.json`, `acceptance.json`, `summaries/`, `state/`, `manifests/`, `validation/`, `parsed/` và `tables/`. Không chỉ gửi ba bảng rời.

`full_assignment_accepted=true` xác nhận pipeline qua đủ phạm vi cá nhân; các cảnh báo `bib_field_conflict` vẫn cần đối chiếu. `resume_all_cached=null` nghĩa là không chạy lượt kiểm tra cache riêng. Receipt ghi `resume_enabled`; khi chạy tiếp output cũ, thời gian đo gồm việc dùng cache, không phải thời gian parsing mới toàn bộ dữ liệu.

Mỗi người chỉ ghi file của mình. Notebook không ghi `group-acceptance.json` dùng chung; TV1 kiểm tra đủ năm output và cùng code SHA trước khi nghiệm thu nhóm. TV5 phụ trách QA, thống kê và viết báo cáo MS2.

In [ ]:
# Bật sau khi parsing xong để tạo ZIP bàn giao riêng cho thành viên.
CREATE_ZIP = False #@param {type:"boolean"}
if CREATE_ZIP:
    import zipfile
    result_dir = OUTPUT_BASE / MEMBER
    if not (result_dir / 'acceptance.json').is_file():
        raise RuntimeError('Chưa có acceptance.json; chạy cell parsing trước.')
    archive = OUTPUT_BASE / f'TV_{MEMBER}_MS2_RESULTS_{RUN_TAG}_{RUN_MODE.lower()}.zip'
    temporary = archive.with_suffix('.zip.partial')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
        for file in sorted(result_dir.rglob('*')):
            if file.is_file():
                bundle.write(file, str(Path(MEMBER) / file.relative_to(result_dir)))
    temporary.replace(archive)
    print('ZIP bàn giao:', archive)
else:
    print('Đặt CREATE_ZIP=True rồi chạy cell này khi muốn đóng gói kết quả.')